# ER＋V2：提前退出／高波动减仓（两个独立变体）
按1→6运行。只复用14个原基准账户，正式批处理计算28个新账户。基准为f8aa6a9登记的ER_V2_REFERENCE，保留真实V2，不是FIXED5 OHLC。源码锁定后不选参。

私有仓库：若Colab不能直接打开，请先授权Colab访问GitHub私有仓库；或下载本.ipynb后在Colab上传。此Notebook不保存凭据；清空任何手工输出后再分享。

2020从2月16日起，2021—2025全年，2026年1—10月；每币每年500U。零手续费／零资金费，退出滑点0.02%，币安USDT永续逐笔代理。2025ETH为S_ALL异常模式。

## 1. 环境和依赖（CPU优先）

In [ ]:
import sys,os,subprocess,shutil,platform
assert sys.version_info >= (3,12), '请使用Python3.12或更高的Colab运行时'
subprocess.run([sys.executable,'-m','pip','-q','install','numpy==2.3.5','pandas==3.0.1','psutil==7.0.0'],check=True)
import psutil
print('Python:',sys.version.split()[0], 'CPU:',os.cpu_count(),platform.machine())
print('RAM available/total GB:',round(psutil.virtual_memory().available/2**30,1),round(psutil.virtual_memory().total/2**30,1))
print('Local disk free GB:',round(shutil.disk_usage('/content').free/2**30,1))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: none. This financial replay uses CPU.')

## 2. 路径、固定提交及私有认证
在Colab Secrets添加`GITHUB_TOKEN`并允许本Notebook读取，或使用隐藏输入。令牌需要该私有仓库读取权限。不会写进代码或Git remote。

In [ ]:
from pathlib import Path
import tempfile,getpass,json
CODE_COMMIT = 'c6d23ea64ef5da600be9bae1dd3d9a1a741c00be'
REPO = Path('/content/er-v2-code-'+CODE_COMMIT[:7])
if (REPO/'.git').exists():
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT, '目录提交不一致；请使用新目录，不覆盖旧任务'
else:
    if REPO.exists() and any(REPO.iterdir()): raise RuntimeError('目标目录非空；换一个新目录')
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub private repository token (hidden): ')
    if not token: raise RuntimeError('缺少私有仓库读取凭据')
    env=os.environ.copy();env['COLAB_GH_TOKEN']=token;env['GIT_TERMINAL_PROMPT']='0'
    with tempfile.TemporaryDirectory() as tmp:
        ask=Path(tmp)/'askpass.py'
        ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["COLAB_GH_TOKEN"])\n')
        ask.chmod(0o700);env['GIT_ASKPASS']=str(ask)
        try:
            REPO.mkdir(parents=True,exist_ok=True)
            for cmd in [['git','init','-q'],['git','remote','add','origin','https://github.com/HITGoose/btc-eth-backtest.git'],['git','-c','credential.helper=','fetch','-q','--depth=1','origin',CODE_COMMIT],['git','checkout','-q','--detach','FETCH_HEAD']]:
                subprocess.run(cmd,cwd=REPO,env=env,check=True,capture_output=True)
        finally:
            env.pop('COLAB_GH_TOKEN',None);del token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT
os.environ['BTC_REPO_ROOT']=str(REPO)
ENTRY=REPO/'outputs/er_v2_colab'
print('Locked implementation:',CODE_COMMIT)
print('Repository:',REPO)

## 3. Drive缓存恢复到本地
Git有基准账本／曲线和代码，没有全部原生K线、逐笔ZIP或2025异常日数组。缺失文件由官方CHECKSUM流程恢复，原生文件还必须匹配冻结SHA。

下列单元格只复制Drive缓存到Colab本地。计算不直接逐笔读Drive。原生行情生成后会保存回Drive；每个完整UTC日的checkpoint和结果也保存回Drive。原始逐笔ZIP留在本地`DATA/archives`；如你已在Drive保存官方ZIP，将相同目录结构放在`btc_eth_market_cache/archives`即可复用。不会自动清理原缓存；空间不足时停止并保留断点。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
DRIVE_DATA=DRIVE/'btc_eth_market_cache'
DRIVE_RESULTS=DRIVE/'btc_eth_er_v2'/CODE_COMMIT[:7]
DATA=Path('/content/btc_eth_market_cache')
RESULTS=Path('/content/er_v2_results_'+CODE_COMMIT[:7])
def restore(source,target):
    if not source.exists(): return
    for p in source.rglob('*'):
        if not p.is_file() or p.name.endswith(('.copying','.tmp','.part')): continue
        dest=target/p.relative_to(source)
        if dest.exists() and dest.stat().st_mtime>=p.stat().st_mtime and dest.stat().st_size==p.stat().st_size: continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.restoring');shutil.copy2(p,temp);temp.replace(dest)
DATA.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
restore(DRIVE_DATA,DATA);restore(DRIVE_RESULTS,RESULTS)
os.environ.update(BTC_DATA_ROOT=str(DATA),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DRIVE_DATA=str(DRIVE_DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS))
print('Local native/raw cache:',DATA)
print('Drive native cache:',DRIVE_DATA)
print('Drive daily checkpoint/results:',DRIVE_RESULTS)
print('Free local GB:',round(shutil.disk_usage(DATA).free/2**30,1))

## 4. 仅8小时试运行与恢复检查
首次可能要下载2020原生K线及所需逐笔日档；试算金融区间仍只有8小时，不是年度账户。随后核验同一个短断点恢复。展示真实V2请求、整笔毛盈亏与止损优先的最小机制检查。CPU/墙钟和阶段耗时写入状态。成功后才执行下一格。

In [ ]:
subprocess.run([sys.executable,str(ENTRY/'smoke.py')],cwd=REPO,check=True)
print('Minimal run complete. Annual tasks have not started.')

## 5. 正式批处理（由你在此启动）
14个年份×币种任务，每任务共享行情的两本独立账户，共28个新账户。自动按可用CPU和内存选择进程数（每任务预留约6GB，最多4）；费用与策略全局变量在独立进程中隔离。年度各500U，原基准14账户不重跑。

Colab断线／重启后重新运行1→3，再运行5；同提交同配置的完成任务跳过，其他从最近已保存UTC日恢复。4是小试算，可跳过再次执行。不能删除指纹检查强行恢复。数据或空间不足只记录该任务错误，其余任务继续。

In [ ]:
# 可选只降低并行数，例如 os.environ['BTC_WORKERS']='1'；不改变策略。
subprocess.run([sys.executable,str(ENTRY/'batch.py')],cwd=REPO,check=True)
print('Batch status:',RESULTS/'batch_status.json')

## 6. 汇总与下载
输出分年／分币／组合指标、独立阶段合计、逐笔匹配、提前退出误杀、仓位归因及集中度。未完成任务明确标UNVERIFIED，不填零。七期简单合计不是跨年复利，历史也不是全新样本外。

下载`results_for_review.zip`交回分析；不含庞大逐笔档案或checkpoint。ZIP中保留完整新账本、分钟权益、配置与日志。

In [ ]:
subprocess.run([sys.executable,str(ENTRY/'report.py')],cwd=REPO,check=True)
restore(RESULTS,DRIVE_RESULTS)
print((RESULTS/'REPORT.md').read_text())
from google.colab import files
files.download(str(RESULTS/'results_for_review.zip'))